# Project 2 Milestone 4: DBSCAN Baseline Clustering on PCA and UMAP Embeddings

This notebook introduces a density-based clustering baseline for Project 2.

The goal is to test whether candidate-rich regions seen in PCA and UMAP diagnostic plots can be identified by an unsupervised clustering method.

DBSCAN is used as an exploratory baseline. Cluster labels should be interpreted as feature-space groupings, not confirmed astrophysical populations.

## Research Question

Can density-based clustering identify candidate-rich regions in PCA or UMAP embedding space?

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

In [ ]:
PCA_PATH = Path('../data/processed/project2_combined_chemo_kinematic_pca_embedding.csv')
UMAP_PATH = Path('../data/processed/project2_combined_chemo_kinematic_umap_embedding.csv')
FIGURE_DIR = Path('../figures')
OUTPUT_DIR = Path('../data/processed')

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pca_df = pd.read_csv(PCA_PATH)
umap_df = pd.read_csv(UMAP_PATH)

pca_df.shape, umap_df.shape

## DBSCAN Helper Functions

In [ ]:
def run_dbscan(dataframe, coordinate_cols, eps, min_samples, label_name):
    result = dataframe.copy()
    X = result[coordinate_cols].to_numpy()
    X_scaled = StandardScaler().fit_transform(X)

    model = DBSCAN(eps=eps, min_samples=min_samples)
    labels = model.fit_predict(X_scaled)
    result[label_name] = labels
    return result


def summarize_dbscan_labels(dataframe, label_col, embedding_name, eps, min_samples):
    labels = dataframe[label_col]
    n_clusters = len(set(labels)) - (1 if -1 in set(labels) else 0)
    noise_fraction = (labels == -1).mean()
    candidate_fraction_noise = dataframe.loc[labels == -1, 'chemo_kinematic_candidate'].mean() if (labels == -1).any() else np.nan

    return {
        'embedding': embedding_name,
        'eps': eps,
        'min_samples': min_samples,
        'n_clusters': n_clusters,
        'noise_fraction': noise_fraction,
        'candidate_fraction_noise': candidate_fraction_noise,
    }


def cluster_summary_table(dataframe, label_col, embedding_name, eps, min_samples):
    rows = []
    for label, group in dataframe.groupby(label_col):
        rows.append({
            'embedding': embedding_name,
            'eps': eps,
            'min_samples': min_samples,
            'cluster_label': int(label),
            'n_stars': len(group),
            'candidate_count': int(group['chemo_kinematic_candidate'].sum()),
            'candidate_fraction': group['chemo_kinematic_candidate'].mean(),
            'metal_poor_fraction': group['metal_poor_candidate'].mean(),
            'high_vtan_fraction': group['high_vtan_candidate'].mean(),
            'mean_feh': group['feh'].mean(),
            'median_feh': group['feh'].median(),
            'mean_galcen_vtot_kms': group['galcen_vtot_kms'].mean(),
            'median_galcen_vtot_kms': group['galcen_vtot_kms'].median(),
        })
    return pd.DataFrame(rows).sort_values(['candidate_fraction', 'candidate_count', 'n_stars'], ascending=[False, False, False])

## Parameter Grid Search

The parameter grid is exploratory. It helps identify whether DBSCAN finds meaningful candidate-rich structures under reasonable settings.

In [ ]:
eps_values = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
min_samples_values = [5, 8, 12, 16]

grid_rows = []

for embedding_name, dataframe, coordinate_cols in [
    ('pca', pca_df, ['combined_chemo_kinematic_pc1', 'combined_chemo_kinematic_pc2']),
    ('umap', umap_df, ['umap_1', 'umap_2']),
]:
    for eps in eps_values:
        for min_samples in min_samples_values:
            label_col = f'{embedding_name}_dbscan_label'
            labelled = run_dbscan(dataframe, coordinate_cols, eps, min_samples, label_col)
            grid_rows.append(summarize_dbscan_labels(labelled, label_col, embedding_name, eps, min_samples))

dbscan_grid_summary = pd.DataFrame(grid_rows)
dbscan_grid_summary

## Save Parameter Grid Summary

In [ ]:
GRID_OUTPUT_PATH = OUTPUT_DIR / 'project2_dbscan_parameter_grid_summary.csv'
dbscan_grid_summary.to_csv(GRID_OUTPUT_PATH, index=False)
GRID_OUTPUT_PATH

## Baseline DBSCAN Configuration

A conservative baseline is selected for both PCA and UMAP embeddings. Later milestones can perform robustness tests across the full parameter grid.

In [ ]:
baseline_eps = 0.25
baseline_min_samples = 8

pca_labelled = run_dbscan(
    pca_df,
    ['combined_chemo_kinematic_pc1', 'combined_chemo_kinematic_pc2'],
    eps=baseline_eps,
    min_samples=baseline_min_samples,
    label_name='pca_dbscan_label',
)

umap_labelled = run_dbscan(
    umap_df,
    ['umap_1', 'umap_2'],
    eps=baseline_eps,
    min_samples=baseline_min_samples,
    label_name='umap_dbscan_label',
)

pca_cluster_summary = cluster_summary_table(pca_labelled, 'pca_dbscan_label', 'pca', baseline_eps, baseline_min_samples)
umap_cluster_summary = cluster_summary_table(umap_labelled, 'umap_dbscan_label', 'umap', baseline_eps, baseline_min_samples)

dbscan_cluster_summary = pd.concat([pca_cluster_summary, umap_cluster_summary], ignore_index=True)
dbscan_cluster_summary.head(20)

## Save Baseline Labels and Cluster Summary

In [ ]:
PCA_LABELS_PATH = OUTPUT_DIR / 'project2_pca_dbscan_labels_baseline.csv'
UMAP_LABELS_PATH = OUTPUT_DIR / 'project2_umap_dbscan_labels_baseline.csv'
CLUSTER_SUMMARY_PATH = OUTPUT_DIR / 'project2_dbscan_cluster_summary_baseline.csv'

pca_labelled.to_csv(PCA_LABELS_PATH, index=False)
umap_labelled.to_csv(UMAP_LABELS_PATH, index=False)
dbscan_cluster_summary.to_csv(CLUSTER_SUMMARY_PATH, index=False)

PCA_LABELS_PATH, UMAP_LABELS_PATH, CLUSTER_SUMMARY_PATH

## Diagnostic Figures

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    umap_labelled['umap_1'],
    umap_labelled['umap_2'],
    c=umap_labelled['umap_dbscan_label'],
    s=18,
    alpha=0.75,
)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Project 2 DBSCAN baseline clusters on UMAP embedding')
cbar = plt.colorbar(scatter)
cbar.set_label('DBSCAN label')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_umap_dbscan_clusters.png', dpi=200)
plt.show()

In [ ]:
candidate_mask = umap_labelled['chemo_kinematic_candidate'].astype(bool)

plt.figure(figsize=(8, 6))
plt.scatter(
    umap_labelled.loc[~candidate_mask, 'umap_1'],
    umap_labelled.loc[~candidate_mask, 'umap_2'],
    s=14,
    alpha=0.35,
    label='Main sample',
)
plt.scatter(
    umap_labelled.loc[candidate_mask, 'umap_1'],
    umap_labelled.loc[candidate_mask, 'umap_2'],
    s=55,
    marker='x',
    alpha=0.95,
    label='Project 1 candidates',
)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.title('Project 2 DBSCAN baseline: candidate overlay on UMAP')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_umap_dbscan_candidate_overlay.png', dpi=200)
plt.show()

In [ ]:
candidate_mask = pca_labelled['chemo_kinematic_candidate'].astype(bool)

plt.figure(figsize=(8, 6))
plt.scatter(
    pca_labelled.loc[~candidate_mask, 'combined_chemo_kinematic_pc1'],
    pca_labelled.loc[~candidate_mask, 'combined_chemo_kinematic_pc2'],
    s=14,
    alpha=0.35,
    label='Main sample',
)
plt.scatter(
    pca_labelled.loc[candidate_mask, 'combined_chemo_kinematic_pc1'],
    pca_labelled.loc[candidate_mask, 'combined_chemo_kinematic_pc2'],
    s=55,
    marker='x',
    alpha=0.95,
    label='Project 1 candidates',
)
plt.xlabel('Combined chemo-kinematic PC1')
plt.ylabel('Combined chemo-kinematic PC2')
plt.title('Project 2 DBSCAN baseline: candidate overlay on PCA')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'project2_pca_dbscan_candidate_overlay.png', dpi=200)
plt.show()

## Milestone 4 Notes

- DBSCAN is used as a baseline density-based clustering method.
- The parameter grid is exploratory and should not be over-interpreted.
- Candidate enrichment in a cluster is more informative than visual separation alone.
- Later milestones should test robustness across feature spaces, scaling choices, and clustering parameters.